modification de la base de donnée d'entrainement a partir ê  modele3v7


In [1]:
import pandas as pd
df = pd.read_csv("modele3v7.csv")
df.columns

Index(['film_id', 'realisateur_id', 'rang', 'title', 'titre_vo', 'realisateur',
       'genre', 'annee', 'pays', 'entrees', 'salles', 'moy_salle',
       'part_marche', 'affiche', 'moyenne_fr_realisateur', 'sortie',
       'distributeur', 'classification', 'acteurs',
       'moyennes_individuelles_acteurs', 'acteur_principal',
       'acteur_secondaire', 'max_moyenne', 'moyennebestactors', 'Pays',
       'cluster_realisateur', 'cluster_acteur_principal',
       'cluster_acteur_secondaire', 'box_office_fr', 'box_office_us',
       'audience', 'budget', 'voix_off', 'demarrage', 'writer', 'languages',
       'sommedesmoyennes', 'duration', 'cluster_acteur',
       'somme_cluster_acteur'],
      dtype='object')

Suppression de features 
    'realisateur_id', 'rang',
    'entrees', 
    'acteur_principal',
    'acteur_secondaire', 'max_moyenne', 'moyennebestactors', 
   
    'box_office_fr', 'box_office_us',
    'budget', 'voix_off'
  

In [3]:
import pandas as pd

# 1) Chargement
df = pd.read_csv("modele3v7.csv")

# 2) Colonnes à supprimer
cols_to_drop = [
    # --- liste initiale ---
    "realisateur_id", "rang", "entrees",
    "acteur_principal", "acteur_secondaire",
    "max_moyenne", "moyennebestactors",
    "box_office_fr", "box_office_us", "budget",
    "voix_off", "salles", "moy_salle", "part_marche",
    # --- ajouts demandés ---
    "titre_vo", "pays", "affiche", "languages",
]

# 3) Suppression (on ignore celles qui pourraient manquer)
df = df.drop(columns=cols_to_drop, errors="ignore")

# 4) Sauvegarde
df.to_csv("modele3v8.csv", index=False)

print(f"✅ Nettoyage terminé : modele3v8.csv créé "
      f"({df.shape[0]} lignes, {df.shape[1]} colonnes).")



✅ Nettoyage terminé : modele3v8.csv créé (7395 lignes, 22 colonnes).


modification de la colonne somme des moyennes des acteurs

In [4]:
import pandas as pd
import ast

# 1) Chargement
df = pd.read_csv("modele3v8.csv")

# 2) Fonction pour extraire et sommer les top 3 moyennes
def sum_top3(moyennes_str):
    try:
        # Convertit la chaîne en liste de dicts
        liste = ast.literal_eval(moyennes_str)
    except (ValueError, SyntaxError, TypeError):
        return 0
    # Récupère toutes les 'moyenne'
    valeurs = [d.get('moyenne', 0) for d in liste if isinstance(d, dict)]
    # Trie par ordre décroissant et somme les 3 premières
    return sum(sorted(valeurs, reverse=True)[:3])

# 3) Application de la fonction
df['sommedesmoyennes'] = df['moyennes_individuelles_acteurs'].apply(sum_top3)

# 4) Sauvegarde
df.to_csv("modele3v9.csv", index=False)

print(f"✅ modèle3v9.csv créé : {df.shape[0]} lignes, {df.shape[1]} colonnes.")


✅ modèle3v9.csv créé : 7395 lignes, 22 colonnes.


creation de la colonne "public"

In [5]:
import pandas as pd
import numpy as np

# 1) Chargement
df = pd.read_csv("modele3v9.csv")

# 2) Normalisation des valeurs vides en NaN
df['audience']       = df['audience'].replace('', np.nan)
df['classification'] = df['classification'].replace('', np.nan)

# 3) Création de la colonne 'public'
#    - si 'audience' est non-null, on la garde
#    - sinon on prend 'classification'
df['public'] = df['audience'].where(df['audience'].notna(), df['classification'])

# 4) Sauvegarde
df.to_csv("modele3v10.csv", index=False)

print(f"✅ modele3v10.csv créé : {df.shape[0]} lignes, {df.shape[1]} colonnes.")


✅ modele3v10.csv créé : 7395 lignes, 23 colonnes.


In [6]:
# Supposons que df est déjà votre DataFrame chargé
# 1) On élimine les NaN (si vous en avez) puis on récupère les uniques
unique_vals = df['public'].dropna().unique()

# 2) (optionnel) Pour les trier et les avoir sous forme de liste Python
unique_list = sorted(unique_vals.tolist())

print("Valeurs uniques dans 'public' :", unique_list)


Valeurs uniques dans 'public' : ['Interdit - 12 ans', 'Interdit - 12 ans avec avertissement', 'Interdit - 16 ans', 'Interdit - 16 ans avec avertissement', 'Interdit - 18 ans', 'Tous publics', 'Tout public', 'Tout public avec avertissement']


In [8]:
import pandas as pd

# 1) Chargement
df = pd.read_csv("modele3v10.csv")

# 2) Dictionnaire de mappage (inchangé)
mapping = {
    "Tous publics":                   "Tout public",
    "Tout public":                    "Tout public",
    "Tout public avec avertissement": "Tout public",
    "Interdit - 12 ans":                     "Interdit - 12 ans",
    "Interdit - 12 ans avec avertissement":  "Interdit - 12 ans",
    "Interdit - 16 ans":                     "Interdit - 16 ans",
    "Interdit - 16 ans avec avertissement":  "Interdit - 16 ans",
    "Interdit - 18 ans":                     "Interdit - 18 ans",
}

# 3) Uniformisation
df['public'] = df['public'].replace(mapping)

# 4) Affichage des valeurs uniques filtrées et triées
unique_vals = df['public'].dropna().unique()      # on enlève les NaN
unique_list = sorted(unique_vals.tolist())       # on peut trier normalement
print("Valeurs après uniformisation :", unique_list)

# 5) Sauvegarde finale
df.to_csv("modele3v11.csv", index=False)
print("✅ modele3v11.csv créé avec les catégories standardisées.")



Valeurs après uniformisation : ['Interdit - 12 ans', 'Interdit - 16 ans', 'Interdit - 18 ans', 'Tout public']
✅ modele3v11.csv créé avec les catégories standardisées.


In [9]:
import pandas as pd

# 1) Chargement du dernier CSV
df = pd.read_csv("modele3v11.csv")

# 2) Suppression des colonnes 'audience' et 'classification'
df = df.drop(columns=["audience", "classification"], errors="ignore")

# 3) Sauvegarde dans un nouveau fichier
df.to_csv("modele3v12.csv", index=False)

print(f"✅ modele3v12.csv créé : {df.shape[0]} lignes, {df.shape[1]} colonnes.")  


✅ modele3v12.csv créé : 7395 lignes, 21 colonnes.


In [10]:
import pandas as pd
import ast
import unicodedata
import re

# 1) Chargement
df = pd.read_csv("modele3v12.csv")
actors_df = pd.read_csv("actors.csv")  # colonnes ['acteur','moyenne','cluster_acteur']

# 2) Fonction de normalisation d’un nom
def normalize(name: str) -> str:
    if not isinstance(name, str):
        return ""
    # décompose accents -> retire les diacritiques
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(ch for ch in name if not unicodedata.combining(ch))
    # minuscules, retire tout ce qui n’est pas lettre ou espace
    name = name.lower()
    name = re.sub(r'[^a-z0-9\s]', '', name)
    # espace unique
    return re.sub(r'\s+', ' ', name).strip()

# 3) Prépare le mapping normalisé {nom_norm: cluster}
actors_df['acteur_norm'] = actors_df['acteur'].map(normalize)
cluster_map = pd.Series(
    actors_df.cluster_acteur.values,
    index=actors_df.acteur_norm
).to_dict()

# 4) Fonction d’agrégation des 3 plus grands clusters
def sum_top3_clusters(acteurs_str):
    try:
        liste = ast.literal_eval(acteurs_str)
    except (ValueError, SyntaxError, TypeError):
        return 0
    clusters = []
    for d in liste:
        nom = d.get("name", "")
        nom_norm = normalize(nom)
        if nom_norm in cluster_map:
            clusters.append(cluster_map[nom_norm])
    return sum(sorted(clusters, reverse=True)[:3]) if clusters else 0

# 5) Création de la colonne
df["somme_cluster_acteur"] = df["acteurs"].apply(sum_top3_clusters)

# 6) Sauvegarde
df.to_csv("modele3v13.csv", index=False)
print(f"✅ modele3v13.csv créé : {df.shape[0]} lignes, {df.shape[1]} colonnes.")


✅ modele3v13.csv créé : 7395 lignes, 21 colonnes.


In [11]:
import pandas as pd

# 1) Chargement
df = pd.read_csv("modele3v13.csv")

# 2) Dictionnaire de renommage
rename_map = {
    "realisateur":                   "directors",
    "annee":                         "year",
    "Pays":                          "country_production",
    "pays":                          "country_production",  # au cas où
    "moyenne_fr_realisateur":        "average_fr_director",
    "acteurs":                       "actors",
    "moyennes_individuelles_acteurs":"moyenne_individuelle_acteurs",
    "sommedesmoyennes":              "total_average_actors",
    "cluster_realisateur":           "popularity_productor",
    "somme_cluster_acteur":          "sum_popularity_actors",
    "demarrage":                     "number_entrances_fr",
}

# 3) Application du renommage
df = df.rename(columns=rename_map)

# 4) Sauvegarde dans un nouveau fichier
df.to_csv("modele3v14.csv", index=False)

print(f"✅ modele3v14.csv créé : {df.shape[0]} lignes, {df.shape[1]} colonnes.")  


✅ modele3v14.csv créé : 7395 lignes, 21 colonnes.


In [12]:
import pandas as pd

# 1) Chargement
df = pd.read_csv("modele3v14.csv")

# 2) Renommage de la colonne supplémentaire
df = df.rename(columns={
    "cluster_acteur": "popularity_actors"
})

# 3) Sauvegarde dans un nouveau fichier
df.to_csv("modele3v15.csv", index=False)

print(f"✅ modele3v15.csv créé : {df.shape[0]} lignes, {df.shape[1]} colonnes.")  


✅ modele3v15.csv créé : 7395 lignes, 21 colonnes.


In [13]:
import pandas as pd

# 1) Chargement
df = pd.read_csv("modele3v15.csv")

# 2) Suppression des colonnes non désirées
cols_to_drop = ["cluster_acteur_principal", "cluster_acteur_secondaire"]
df = df.drop(columns=cols_to_drop, errors="ignore")

# 3) Sauvegarde dans un nouveau fichier
df.to_csv("modele3v16.csv", index=False)

print(f"✅ modele3v16.csv créé : {df.shape[0]} lignes, {df.shape[1]} colonnes.")  


✅ modele3v16.csv créé : 7395 lignes, 19 colonnes.


In [15]:
import pandas as pd
import numpy as np
import re
import ast

# 1) Charger le CSV en forçant actors & directors en str
df = pd.read_csv(
    "modele3v16.csv",
    dtype={
        "actors": str,
        "directors": str,
        "distributeur": str
    }
)

# 2) Nettoyer number_entrances_fr
def to_int(txt):
    if isinstance(txt, str):
        t = re.sub(r"[^\d]", "", txt)
        return int(t) if t else np.nan
    return int(txt) if not pd.isna(txt) else np.nan

df["number_entrances_fr"] = df["number_entrances_fr"].apply(to_int)

# 3) Remplacer les NaN de actors/directors/distributeur par des valeurs valides
df["actors"]       = df["actors"].fillna("[]")
df["directors"]    = df["directors"].fillna("[]")
df["distributeur"] = df["distributeur"].fillna("Unknown")

# 4) Fonction de parsing sûre
def parse_names(json_str):
    try:
        lst = ast.literal_eval(json_str)
        # Si c’est bien une liste de dicts [{'name':...},…]
        return [d.get("name") for d in lst if isinstance(d, dict) and "name" in d]
    except (ValueError, SyntaxError):
        # Chaîne malformée ou autre → on renvoie liste vide
        return []

# 5) Appliquer au DataFrame
df["actors_list"]    = df["actors"].apply(parse_names)
df["directors_list"] = df["directors"].apply(parse_names)
df["distrib_name"]   = df["distributeur"]

# 6) Vérification
print(df[["actors_list", "directors_list", "distrib_name"]].head())
print("Films avec entrée connue :", df["number_entrances_fr"].notna().sum(), "/", len(df))



                                         actors_list directors_list  \
0                       [Harrison Ford, Adam Driver]             []   
1                   [Ewan McGregor, Natalie Portman]             []   
2            [Jamel Debbouze, Jean Reno, Seth Rogen]             []   
3  [Frédéric Diefenthal, Samy Nacéri, Marion Coti...             []   
4            [Daniel Craig, Gad Elmaleh, Simon Pegg]             []   

           distrib_name  
0  Walt Disney Pictures  
1      20th Century Fox  
2  Walt Disney Pictures  
3         ARP Selection  
4         Sony Pictures  
Films avec entrée connue : 7395 / 7395


In [17]:
num_cols = [
    "average_fr_director", "total_average_actors", "popularity_productor",
    "popularity_actors", "sum_popularity_actors", "duration", "year",
    "month", "quarter", "vacances"
]


In [18]:
# On part du principe que num_cols existe déjà
num_cols.extend([
    'actors_avg_top3', 'actors_count_top3',
    'director_avg',    'director_count',
    'dist_avg',        'dist_count'
])


In [19]:
import numpy as np
from collections import defaultdict

# 1) Construire les historiques de box‑office par entité
actor_hist = defaultdict(list)
dir_hist   = defaultdict(list)
dist_hist  = defaultdict(list)

for entr, actors, dirs, dist in zip(
    df['number_entrances_fr'],
    df['actors_list'],
    df['directors_list'],
    df['distrib_name']
):
    if np.isnan(entr):
        continue
    for a in actors:
        actor_hist[a].append(entr)
    for d in dirs:
        dir_hist[d].append(entr)
    dist_hist[dist].append(entr)

# 2) Calculer moyenne et compte
actor_avg   = {a: np.mean(vals) for a, vals in actor_hist.items()}
actor_count = {a: len(vals)    for a, vals in actor_hist.items()}

dir_avg     = {d: np.mean(vals) for d, vals in dir_hist.items()}
dir_count   = {d: len(vals)    for d, vals in dir_hist.items()}

dist_avg    = {d: np.mean(vals) for d, vals in dist_hist.items()}
dist_count  = {d: len(vals)    for d, vals in dist_hist.items()}

# 3) Fonctions pour extraire top‑k et total
def topk_mean(names, mapping, k=3):
    vals = sorted((mapping.get(n, 0) for n in names), reverse=True)[:k]
    return np.mean(vals) if vals else 0

def topk_sum(names, mapping, k=3):
    vals = sorted((mapping.get(n, 0) for n in names), reverse=True)[:k]
    return np.sum(vals) if vals else 0

# 4) Injecter dans df
df['actors_avg_top3']   = df['actors_list'].apply(lambda x: topk_mean(x, actor_avg,   k=3))
df['actors_count_top3'] = df['actors_list'].apply(lambda x: topk_mean(x, actor_count, k=3))

df['director_avg']      = df['directors_list'].apply(lambda x: topk_mean(x, dir_avg,   k=1))
df['director_count']    = df['directors_list'].apply(lambda x: topk_mean(x, dir_count, k=1))

df['dist_avg']          = df['distrib_name'].map(dist_avg).fillna(0)
df['dist_count']        = df['distrib_name'].map(dist_count).fillna(0)

# 5) Mettre à jour la liste num_cols pour le pipeline
num_cols.extend([
    'actors_avg_top3', 'actors_count_top3',
    'director_avg', 'director_count',
    'dist_avg', 'dist_count'
])

In [20]:
df.to_csv("modele3v17.csv", index=False)

In [1]:
# train_boxoffice_two_stage.py
# -*- coding: utf-8 -*-
"""
Pipeline en deux étapes optimisée :
1. Classification des films « hits » (top 20 %)
2. Régression du nombre d’entrées (log1p) sur les hits uniquement

Les colonnes vides directors_list, director_avg, director_count ont été supprimées.
"""
import re, json, pickle, pathlib, warnings
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (roc_auc_score, recall_score,
                             root_mean_squared_error as rmse,
                             mean_absolute_error, r2_score)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from category_encoders import CountEncoder, TargetEncoder
import joblib


warnings.filterwarnings("ignore")

# 1) Chargement et préparation
# ------------------------------------------------------------
# On part du principe que modele3v17.csv contient désormais :
# actors_list, distrib_name, dist_avg, dist_count, acteurs stats

df = pd.read_csv("modele3v17.csv", dtype={"actors":str, "distributeur":str})

# Conversion cible
import ast

def to_int(txt):
    if isinstance(txt, str):
        t = re.sub(r"[^\d]", "", txt)
        return int(t) if t else np.nan
    return int(txt) if not pd.isna(txt) else np.nan

df['number_entrances_fr'] = df['number_entrances_fr'].apply(to_int)

# Date features
df['sortie']   = pd.to_datetime(df['sortie'], format="%d/%m/%Y", errors='coerce')
df['month']    = df['sortie'].dt.month
df['quarter']  = df['sortie'].dt.quarter
df['vacances'] = df['month'].isin([7,8,12,1,2]).astype('int8')

# Cibles

y_raw = df['number_entrances_fr'].values
threshold = np.nanpercentile(y_raw, 80)
mask_hit = (y_raw >= threshold).astype(int)
y_log = np.log1p(y_raw)

# 2) Définition des features
# ------------------------------------------------------------
# Colonnes numériques enrichies (sans directors)
num_cols = [
    'average_fr_director', 'total_average_actors', 'popularity_productor',
    'popularity_actors', 'sum_popularity_actors', 'duration', 'year',
    'month', 'quarter', 'vacances',
    'actors_avg_top3', 'actors_count_top3',
    'dist_avg', 'dist_count'
]
# Colonnes catégorielles
cat_short = ['genre','country_production','public','distributeur']
cat_long  = ['actors','writer']

# 3) Pipeline de pré-processing
# ------------------------------------------------------------
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ce = CountEncoder(cols=cat_long)
te = TargetEncoder(cols=cat_long, smoothing=0.3)
preproc = ColumnTransformer([
    ('num',        'passthrough', num_cols),
    ('cat_short',  ord_enc,       cat_short),
    ('count_long', ce,            cat_long),
    ('target_long',te,            cat_long),
], remainder='drop')

# 4) Métriques helper
# ------------------------------------------------------------
def inv_log(x): return np.expm1(x)

def fold_reg_metrics(y_true_log, y_pred_log):
    rmse_log = rmse(y_true_log, y_pred_log)
    yt = inv_log(y_true_log); yp = inv_log(y_pred_log)
    rmse_lin = rmse(yt, yp)
    mae_lin  = mean_absolute_error(yt, yp)
    r2_lin   = r2_score(yt, yp)
    rho, _   = spearmanr(yt, yp)
    k = max(int(len(yp)*0.10),1)
    capture = yt[np.argsort(-yp)[:k]].sum() / yt.sum()
    return rmse_log, rmse_lin, mae_lin, r2_lin, rho, capture

# 5) Cross-validation chrono 2-étapes
# ------------------------------------------------------------
clf_scores = []
reg_scores = []
tss = TimeSeriesSplit(n_splits=5)

for tr_idx, te_idx in tss.split(df):
    X_tr, X_te = df.iloc[tr_idx], df.iloc[te_idx]
    y_hit_tr, y_hit_te = mask_hit[tr_idx], mask_hit[te_idx]
    y_log_tr, y_log_te = y_log[tr_idx], y_log[te_idx]

    # Pré‑processing complet
    X_tr_t = preproc.fit_transform(X_tr, y_log_tr)
    X_te_t = preproc.transform(X_te)

    # Étape 1: classification des hits
    if np.any(y_hit_tr == 1) and np.any(y_hit_te == 1):
        clf = RandomForestClassifier(
            n_estimators=200, max_depth=8,
            class_weight='balanced', random_state=42
        )
        clf.fit(X_tr_t, y_hit_tr)
        prob = clf.predict_proba(X_te_t)[:,1]
        auc = roc_auc_score(y_hit_te, prob)
        # seuil pour recall@70%
        thresh = np.percentile(prob[y_hit_te==1], 70)
        pred_hit = (prob >= thresh).astype(int)
        rec = recall_score(y_hit_te, pred_hit)
        clf_scores.append((auc, rec))
    else:
        continue  # skip fold sans hits

    # Étape 2: régression sur hits
    hit_tr = np.where(y_hit_tr == 1)[0]
    hit_te = np.where(pred_hit == 1)[0]
    if hit_tr.size and hit_te.size:
        X_r_tr = X_tr_t[hit_tr]; y_r_tr = y_log_tr[hit_tr]
        X_r_te = X_te_t[hit_te]; y_r_te = y_log_te[hit_te]
        model = lgb.LGBMRegressor(
            objective='regression_l2', metric='rmse', device='gpu'
        )
        model.fit(X_r_tr, y_r_tr)
        preds = model.predict(X_r_te)
        reg_scores.append(fold_reg_metrics(y_r_te, preds))

# Agrégation CV
mean_auc = np.mean([a for a,_ in clf_scores])
mean_rec = np.mean([r for _,r in clf_scores])
mean_reg = np.mean(reg_scores, axis=0)
print(f"CV Classif => AUC: {mean_auc:.3f}, Recall: {mean_rec:.3f}")
print(
    f"CV Régr => RMSE_log: {mean_reg[0]:.3f}, RMSE_lin: {mean_reg[1]:,.0f}, \
" +
    f"MAE: {mean_reg[2]:,.0f}, R2: {mean_reg[3]:.3f}, Top10%: {mean_reg[5]*100:.1f}%"
)

# 6) Entraînement final\# ------------------------------------------------------------
X_full_t = preproc.fit_transform(df, y_log)
clf_final = RandomForestClassifier(
    n_estimators=200, max_depth=8,
    class_weight='balanced', random_state=42
)
clf_final.fit(X_full_t, mask_hit)
hit_full = np.where(mask_hit == 1)[0]
X_rf = X_full_t[hit_full]; y_rf = y_log[hit_full]
reg_final = lgb.LGBMRegressor(
    objective='regression_l2', metric='rmse', device='gpu'
)
reg_final.fit(X_rf, y_rf)

# 7) Sauvegarde
pathlib.Path('models').mkdir(exist_ok=True)
joblib.dump(clf_final, 'models/clf_hits.joblib')
joblib.dump(reg_final, 'models/reg_hits.joblib')
with open('models/two_stage_metrics.json','w') as f:
    json.dump({
        'auc': round(mean_auc,4),
        'recall': round(mean_rec,4),
        'rmse_log': round(mean_reg[0],4),
        'rmse_lin': int(mean_reg[1]),
        'mae': int(mean_reg[2])
    }, f, indent=2)
print("\n✅ Pipeline deux‑étapes mise à jour et sauvegardée !")


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1032
[LightGBM] [Info] Number of data points in the train set: 839, number of used features: 21
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3060 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 18 dense feature groups (0.02 MB) transferred to GPU in 0.000716 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 15.529946
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1212
[LightGBM] [Info] Number of data points in the train set: 1476, number of used features: 21
[LightGBM] [Inf